# Week 3: Aggregations in MongoDB


We reproduce the steps below for you to install MongoDB. It is quite involved.

**If you are struggling to get MongoDB working in Google Colab, you can try this tutorial on your local machine. The experience will be less traumatic. All these instructions are given in Installation and Quick start for MongoDB on Brightspace in the folder About the tools.**

In [ ]:
!sudo apt update

In [ ]:
!sudo apt upgrade

In [ ]:
!sudo apt install gnupg2

In [ ]:
!wget -nc https://www.mongodb.org/static/pgp/server-6.0.asc

In [ ]:
!cat server-6.0.asc | gpg --dearmor | sudo tee /etc/apt/keyrings/mongodb.gpg > /dev/null

In [ ]:
!sudo sh -c 'echo "deb [ arch=amd64,arm64 signed-by=/etc/apt/keyrings/mongodb.gpg] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/6.0 multiverse" >> /etc/apt/sources.list.d/mongo.list'

In [ ]:
!sudo apt update

In [ ]:
!sudo apt install mongodb-org

Let's create a database directory for MongoDB server.

In [ ]:
!mkdir -p data/db

In [ ]:
!mongod --nojournal --dbpath data/db --fork --logpath data/db/log

In this tutorial, we will work on aggregations.

More on aggregations here:
https://www.mongodb.com/docs/manual/aggregation/

**Aggregations have replaced MapReduce, which will be one of major topics in big data batch processing in this module.**

Aggregation operations process multiple documents and return computed results. You can use aggregation operations to:

1. Group values from multiple documents together.
2. Perform operations on the grouped data to return a single result.
3. Analyze data changes over time.

Aggregation pipelines, which are the preferred method for performing aggregations.

Now start the MongoDB shell by typing **mongosh**. You will be shown the following output and a prompt where you can type MongoDB commands.

```python
Current Mongosh Log ID:	651c72e470637a022abe2c6a
Connecting to:		mongodb://127.0.0.1:27017/?directConnection=true&serverSelectionTimeoutMS=2000&appName=mongosh+2.0.1
Using MongoDB:		6.0.10
Using Mongosh:		2.0.1

For mongosh info see: https://docs.mongodb.com/mongodb-shell/


To help improve our products, anonymous usage data is collected and sent to MongoDB periodically (https://www.mongodb.com/legal/privacy-policy).
You can opt-out by running the disableTelemetry() command.

 ------
   The server generated these startup warnings when booting
   2023-10-03T19:58:21.241+00:00: Access control is not enabled for the database. Read and write access to data and configuration is unrestricted
   2023-10-03T19:58:21.241+00:00: You are running this process as the root user, which is not recommended
   2023-10-03T19:58:21.241+00:00: This server is bound to localhost. Remote systems will be unable to connect to this server. Start the server with --bind_ip <address> to specify which IP addresses it should serve responses from, or with --bind_ip_all to bind to all interfaces. If this behavior is desired, start the server with --bind_ip 127.0.0.1 to disable this warning
   2023-10-03T19:58:21.241+00:00: vm.max_map_count is too low
test>
```

**Now you can type DML operations at the prompt. When you type, you will see dots. Probably there is a way to see your input.**

Let us create a collection of doucments of pizza orders.

If you are working on your local machine, type the following in the MongoDB shell.

```python
test> db.orders.insertMany( [
   { _id: 0, name: "Pepperoni", size: "small", price: 19,
     quantity: 10, date: ISODate( "2021-03-13T08:14:30Z" ) },
   { _id: 1, name: "Pepperoni", size: "medium", price: 20,
     quantity: 20, date : ISODate( "2021-03-13T09:13:24Z" ) },
   { _id: 2, name: "Pepperoni", size: "large", price: 21,
     quantity: 30, date : ISODate( "2021-03-17T09:22:12Z" ) },
   { _id: 3, name: "Cheese", size: "small", price: 12,
     quantity: 15, date : ISODate( "2021-03-13T11:21:39.736Z" ) },
   { _id: 4, name: "Cheese", size: "medium", price: 13,
     quantity:50, date : ISODate( "2022-01-12T21:23:13.331Z" ) },
   { _id: 5, name: "Cheese", size: "large", price: 14,
     quantity: 10, date : ISODate( "2022-01-12T05:08:13Z" ) },
   { _id: 6, name: "Vegan", size: "small", price: 17,
     quantity: 10, date : ISODate( "2021-01-13T05:08:13Z" ) },
   { _id: 7, name: "Vegan", size: "medium", price: 18,
     quantity: 10, date : ISODate( "2021-01-13T05:10:13Z" ) }
] )
```

**Output:**
```python
{
  acknowledged: true,
  insertedIds: { '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7 }
}
```

In [ ]:
# Create the orders collection above.
!mongosh

The following aggregation pipeline example contains two stages and returns the total order quantity of medium size pizzas grouped by pizza name:

```python
test> db.orders.aggregate( [

   // Stage 1: Filter pizza order documents by pizza size
   {
      $match: { size: "medium" }
   },

   // Stage 2: Group remaining documents by pizza name and calculate total quantity
   {
      $group: { _id: "$name", totalQuantity: { $sum: "$quantity" } }
   }

] )
```

**Output:**
```python
[
  { _id: 'Pepperoni', totalQuantity: 20 },
  { _id: 'Cheese', totalQuantity: 50 },
  { _id: 'Vegan', totalQuantity: 10 }
]
```

The **$match** stage:

1. Filters the pizza order documents to pizzas with a size of medium.
2. Passes the remaining documents to the **$group** stage.

The **$group** stage:

1. Groups the remaining documents by pizza name.
2. Uses **$sum** to calculate the total order quantity for each pizza name. The total is stored in the totalQuantity field returned by the aggregation pipeline.

In [ ]:
# Run the aggregation pipeline to get the total order quantity.
!mongosh

Calculate the total order value and average order quantity using the aggregation pipeline below:

```python
db.orders.aggregate( [

   // Stage 1: Filter pizza order documents by date range
   {
      $match:
      {
         "date": { $gte: new ISODate( "2020-01-30" ), $lt: new ISODate( "2022-01-30" ) }
      }
   },

   // Stage 2: Group remaining documents by date and calculate results
   {
      $group:
      {
         _id: { $dateToString: { format: "%Y-%m-%d", date: "$date" } },
         totalOrderValue: { $sum: { $multiply: [ "$price", "$quantity" ] } },
         averageOrderQuantity: { $avg: "$quantity" }
      }
   },

   // Stage 3: Sort documents by totalOrderValue in descending order
   {
      $sort: { totalOrderValue: -1 }
   }

 ] )
 ```

**Output:**

```python
 [
  { _id: '2022-01-12', totalOrderValue: 790, averageOrderQuantity: 30 },
  { _id: '2021-03-13', totalOrderValue: 770, averageOrderQuantity: 15 },
  { _id: '2021-03-17', totalOrderValue: 630, averageOrderQuantity: 30 },
  { _id: '2021-01-13', totalOrderValue: 350, averageOrderQuantity: 10 }
]
```

The \$match stage:

1. Filters the pizza order documents to those in a date range specified using \$gte and \$lt.
2. Passes the remaining documents to the $group stage.

The \$group stage:

1. Groups the documents by date using \$dateToString.
2. For each group, calculates:
   Total order value using \$sum and \$multiply.
   Average order quantity using \$avg.

Passes the grouped documents to the \$sort stage.

The \$sort stage:

1. Sorts the documents by the total order value for each group in descending order (-1).
2. Returns the sorted documents.

**Do the following exercises:**

In [ ]:
# Run the aggregation pipeline to get the total order value and average order quantity.
!mongosh

Filter pizza order documents by pizza name 'Pepperoni' and calculate **max** quantity.

In [ ]:
#Solve
!mongosh

Group documents by pizza name and calculate **max** quantity for each pizza name.

In [ ]:
#Solve
!mongosh

Group documents by pizza name and calculate **min** quantity for each pizza name.

In [ ]:
#Solve
!mongosh